# Phase 3: Optimization Analysis

In this notebook, we use N-BEATS demand forecasts to optimize shipping allocation using integer linear programming

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import logging
import warnings
warnings.filterwarnings('ignore')

from src.utils.data_loader import DataLoader
from src.utils.data_cleaner import DataCleaner
from src.forecasting.nbeats_model import NBeatsModel
from src.forecasting.model_trainer import ModelTrainer
from src.optimization.ilp_model import ILPShippingModel
from src.optimization.solver import Solver
from src.optimization.baselines import all_standard_baseline
from src.optimization.optimization_comparison import OptimizationComparison
"""from src.utils.visualization import (
    plot_allocation_comparison,
    plot_delivery_time_comparison,
    plot_cost_breakdown,
    plot_service_level,
)"""
from config import DATA_RAW, DATAFILE, TRAIN_WEEKS, TEST_WEEKS, FORECAST_HORIZON

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("Imports successful")

## Load Data and Generate Forecast

In [ ]:
# Load and prepare data
loader = DataLoader(DATA_RAW / DATAFILE)
raw_data = loader.load()

cleaner = DataCleaner(raw_data, frequency='W')
weekly_demand = cleaner.aggregate_by_frequency()
weekly_demand = cleaner.handle_missing_values()
weekly_demand = cleaner.remove_outliers(method='iqr')

train, test = cleaner.train_test_split(train_weeks=TRAIN_WEEKS, test_weeks=TEST_WEEKS)

print(f"  Data loaded: {len(train)} train, {len(test)} test")
print(f"  Train range: {train.index[0].date()} to {train.index[-1].date()}")
print(f"  Test range: {test.index[0].date()} to {test.index[-1].date()}")

In [ ]:
# Train N-BEATS for demand forecast
nbeats = NBeatsModel(
    horizon=FORECAST_HORIZON,
    lookback=8,
    epochs=50,
    batch_size=32,
)

trainer = ModelTrainer(nbeats, train, test)
metrics = trainer.train()

print(f"\nN-BEATS trained successfully")
print(f"  SMAPE: {metrics['smape']:.2f}%")
print(f"  MAE: {metrics['mae']:.2f}")

In [ ]:
# Generate forecast
forecast = nbeats.predict(FORECAST_HORIZON)
forecast_demand = int(forecast.sum())

print(f"\nForecast for Optimization:")
print(f"  Total units (4 weeks): {forecast_demand}")

# Display forecast
forecast_df = pd.DataFrame({
    'Week': range(1, FORECAST_HORIZON + 1),
    'Forecast': [int(x) for x in forecast.values]
})
print(f"\n{forecast_df.to_string(index=False)}")

## Set Up Optimization Problem

In [ ]:
# define shipping modes and parameters
shipping_modes = ["First Class", "Same Day", "Second Class", "Standard Class"]

delivery_times = {
    "First Class": 2.0,
    "Same Day": 1.0,
    "Second Class": 3.0,
    "Standard Class": 4.0,
}

# Cost per unit
costs = {
    "First Class": 1.5,
    "Same Day": 2.5,
    "Second Class": 1.0,
    "Standard Class": 0.8,
}

# Capacity constraints
capacities = {
    "First Class": 650.0,
    "Same Day": 400.0,
    "Second Class": 1500.0,
    "Standard Class": 3000.0,
}

budget = 5500
fast_service_ratio = 0.10

print("Shipping Modes Configuration:")
print("-" * 70)
config_df = pd.DataFrame({
    'Mode': shipping_modes,
    'Delivery Days': [delivery_times[m] for m in shipping_modes],
    'Cost/Unit': [costs[m] for m in shipping_modes],
    'Capacity': [capacities[m] for m in shipping_modes],
})
print(config_df.to_string(index=False))
print("-" * 70)
print(f"Budget Limit: ${budget}")
print(f"Service Level: {fast_service_ratio*100:.0f}% minimum fast shipping")

## Build and solve ILP Model

In [ ]:
# Create ILP model
ilp_model = ILPShippingModel(
    demand_forecast=forecast_demand,
    shipping_modes=shipping_modes,
    delivery_times=delivery_times,
    costs=costs,
    capacities=capacities,
    budget=budget,
    fast_service_ratio=fast_service_ratio,
)

# Build model
ilp_model.build_model()
print("ILP model built")

# Solve
solver = Solver(time_limit=60)
ilp_solution = solver.solve(ilp_model)

if ilp_solution:
    print(f"Optimal solution found\n")
else:
    print("No optimal solution found")

In [ ]:
# Display optimal allocation
print("OPTIMAL ALLOCATION (ILP Solution)")
print("=" * 80)

allocation_df = pd.DataFrame({
    'Mode': shipping_modes,
    'Units': [ilp_solution.get(m, 0) for m in shipping_modes],
    'Cost/Unit': [costs[m] for m in shipping_modes],
    'Days/Unit': [delivery_times[m] for m in shipping_modes],
})

allocation_df['Total Cost'] = allocation_df['Units'] * allocation_df['Cost/Unit']
allocation_df['Total Days'] = allocation_df['Units'] * allocation_df['Days/Unit']

print(allocation_df.to_string(index=False))
print("=" * 80)

metrics = solver.get_metrics(ilp_model, ilp_solution)

print(f"\n📊 OPTIMIZATION METRICS:")
print(f"  Total Units: {forecast_demand}")
print(f"  Total Cost: ${metrics['total_cost']}")
print(f"  Total Delivery Days: {metrics['total_delivery_days']}")
print(f"  Average Days/Unit: {metrics['avg_delivery_days']}")
print(f"  Fast Units (First Class + Same Day): {metrics['fast_units']}")
print(f"  Fast Service %: {metrics['fast_ratio']}%")
print(f"  Budget Used: {metrics['budget_used']}% of ${budget}")

## Baseline Comparison

In [ ]:
# Generate baseline strategy
baseline_solution = all_standard_baseline(
    forecast_demand,
    shipping_modes,
    capacities,
    costs,
    budget,
)

print("📦 BASELINE ALLOCATION (All Standard Class)")
print("=" * 80)

baseline_df = pd.DataFrame({
    'Mode': shipping_modes,
    'Units': [baseline_solution.get(m, 0) for m in shipping_modes],
    'Cost/Unit': [costs[m] for m in shipping_modes],
    'Days/Unit': [delivery_times[m] for m in shipping_modes],
})

baseline_df['Total Cost'] = baseline_df['Units'] * baseline_df['Cost/Unit']
baseline_df['Total Days'] = baseline_df['Units'] * baseline_df['Days/Unit']

print(baseline_df.to_string(index=False))
print("=" * 80)

baseline_metrics = solver.get_metrics(ilp_model, baseline_solution)

print(f"\n📊 BASELINE METRICS:")
print(f"  Total Cost: ${baseline_metrics['total_cost']}")
print(f"  Total Delivery Days: {baseline_metrics['total_delivery_days']}")
print(f"  Average Days/Unit: {baseline_metrics['avg_delivery_days']}")
print(f"  Fast Service %: {baseline_metrics['fast_ratio']}%")

## Comparison

In [ ]:
# Compare solutions
assert(baseline_solution is not None)
assert(ilp_solution is not None)

comparison = OptimizationComparison()
results = comparison.compare(
    ilp_solution,
    {"Baseline": baseline_solution},
    ilp_model,
    solver,
)

print("\nOPTIMIZATION vs BASELINE COMPARISON")
print("=" * 100)
print(results.to_string(index=False))
print("=" * 100)

In [ ]:
# Calculate improvements
ilp_days = metrics['total_delivery_days']
baseline_days = baseline_metrics['total_delivery_days']
improvement = comparison.get_improvement(ilp_days, baseline_days)

cost_diff = baseline_metrics['total_cost'] - metrics['total_cost']

print(f"\nKEY IMPROVEMENTS:")
print(f"  Delivery Time: {ilp_days} vs {baseline_days} days ({improvement:.1f}% better)")
print(f"  Cost Difference: ${cost_diff:+.2f}")
print(f"  Service Level: {metrics['fast_ratio']:.1f}% vs {baseline_metrics['fast_ratio']:.1f}% fast shipping")

## Visualizations

In [ ]:
# Allocation comparison
allocations = {
    'ILP Optimal': ilp_solution,
    'Baseline All Standard': baseline_solution,
}

fig = plot_allocation_comparison(allocations, shipping_modes)
fig.show()
print("✓ Allocation comparison saved")

In [ ]:
# Delivery time comparison
fig = plot_delivery_time_comparison(results)
fig.show()
print("✓ Delivery time comparison saved")

In [ ]:
# Cost breakdown
fig = plot_cost_breakdown(ilp_solution, costs)
fig.show()
print("✓ Cost breakdown saved")

In [ ]:
# Service level compliance
fig = plot_service_level(results)
fig.show()
print("✓ Service level compliance saved")